# RECIP-E — Interpretador (front-end) em Scheme

Projeto da disciplina **MC346 — Paradigmas de Programação** (UNICAMP).

Este é o **notebook único** do grupo: todas as etapas do processamento da DSL
**RECIP-E** vivem aqui dentro, em ordem de pipeline — léxico → sintático →
impressão/validação. Kernel: **Calysto Scheme**.

> **Estado: Dia 1 (Sáb 27/06)** — a seção **1. Lexer** (Pessoa A) já está
> implementada. As seções **2. Parser** (Pessoa B) e **3. Pretty-print +
> Validação** (Pessoa C) estão reservadas e serão preenchidas pelos donos.
>
> Para conferir tudo: **Kernel → Restart & Run All**.

## 0. Setup / Contrato do token

Tudo no projeto conversa por meio de **tokens**. Combinamos um formato simples:

```
token = (tipo lexema linha)
```

- **tipo**  — um símbolo: `NUMBER STRING IDENT KEYWORD VERB UNIT PREP PONT`
- **lexema** — o texto original (string)
- **linha** — número da linha no código-fonte (útil para mensagens de erro)

As funções abaixo (`faz-token`, `tipo-tok`, `lexema-tok`, `linha-tok`) são a
**interface** que a Pessoa B (parser) vai usar para ler os tokens.

In [ ]:
;; ---- Contrato do token: (tipo lexema linha) ----
(define (faz-token tipo lexema linha) (list tipo lexema linha))
(define (tipo-tok   t) (car   t))
(define (lexema-tok t) (cadr  t))
(define (linha-tok  t) (caddr t))

;; ---- Tabelas de palavras reservadas da linguagem ----
(define palavras-bloco
  (list "RECEITA" "INGREDIENTES" "UTENSILIOS" "METODOS"))

(define palavras-controle
  (list "ESTE" "é" "AGUARDAR" "VERIFICAR" "SE" "ESTA"
        "CONTINUAR" "SENAO" "AO_MESMO_TEMPO"
        "GRAUS" "MINUTOS" "HORAS" "SEGUNDOS"))

(define preposicoes
  (list "EM" "COM" "USANDO" "A" "POR" "ATE" "NO_ESTILO"))

(define verbos
  (list "ADICIONAR" "MISTURAR" "BATER" "TEMPERAR" "ASSAR" "FRITAR"
        "COZINHAR" "REFOGAR" "GRELHAR" "CORTAR" "RALAR" "ESCORRER"
        "COLOCAR" "DECORAR"))

(define unidades
  (list "g" "kg" "ml" "l" "unidade" "colher" "colher_cha"
        "xcara" "pitada" "a_gosto" "faca"))

;; verifica se uma palavra esta numa lista (devolve #t ou #f)
(define (esta-em? palavra lista)
  (if (member palavra lista) #t #f))

;; descobre o tipo de uma palavra ja lida pelo lexer
(define (classifica palavra)
  (cond
    ((esta-em? palavra palavras-bloco)    'KEYWORD)
    ((esta-em? palavra palavras-controle) 'KEYWORD)
    ((esta-em? palavra preposicoes)       'PREP)
    ((esta-em? palavra verbos)            'VERB)
    ((esta-em? palavra unidades)          'UNIT)
    (else                                 'IDENT)))

## 1. Lexer (Pessoa A)

O lexer transforma o texto da receita numa **lista de tokens**. Nesta entrega
(Dia 1) ele já reconhece números, strings, identificadores, palavras reservadas
e ignora comentários. A indentação (`INDENT`/`DEDENT`) entra no Dia 2.

### 1.1 Predicados auxiliares

Funções pequenas para classificar caracteres. Definimos `TAB` e `RET` via
`integer->char` para não depender de literais de caractere especiais.

In [ ]:
;; alguns caracteres "invisiveis"
(define TAB (integer->char 9))    ; tabulacao
(define RET (integer->char 13))   ; carriage return

;; e um digito de 0 a 9?
(define (eh-digito? c)
  (and (char>=? c #\0) (char<=? c #\9)))

;; e uma letra (A-Z, a-z), underline ou letra acentuada (é, ç, ...)?
(define (eh-letra? c)
  (let ((code (char->integer c)))
    (or (and (>= code 65) (<= code 90))     ; A-Z
        (and (>= code 97) (<= code 122))    ; a-z
        (= code 95)                         ; _
        (>= code 128))))                    ; acentuadas

;; e um espaco em branco (sem contar a quebra de linha)?
(define (eh-espaco? c)
  (or (char=? c #\space) (char=? c TAB) (char=? c RET)))

### 1.2 Remoção de comentários

RECIP-E aceita comentário de linha `// ...` e de bloco `/* ... */`. Antes de
tokenizar, trocamos os comentários por nada — **mas preservamos as quebras de
linha**, para que o número de linha continue certo. Também tomamos cuidado para
não apagar um `//` que esteja **dentro de uma string** (`"..."`).

A função varre o texto caractere a caractere usando um pequeno *estado*:
`normal`, `string`, `linha` (comentário de linha) ou `bloco` (comentário de bloco).

In [ ]:
;; remove comentarios // e /* */, preservando quebras de linha e strings
(define (remove-comentarios s)
  (let ((n (string-length s)))
    (define (loop i estado acc)
      (if (>= i n)
          (list->string (reverse acc))
          (let ((c (string-ref s i)))
            (cond
              ;; ----- texto normal -----
              ((eq? estado 'normal)
               (cond
                 ((char=? c #\")
                  (loop (+ i 1) 'string (cons c acc)))
                 ((and (char=? c #\/) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\/))
                  (loop (+ i 2) 'linha acc))
                 ((and (char=? c #\/) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\*))
                  (loop (+ i 2) 'bloco acc))
                 (else
                  (loop (+ i 1) 'normal (cons c acc)))))
              ;; ----- dentro de uma string -----
              ((eq? estado 'string)
               (if (char=? c #\")
                   (loop (+ i 1) 'normal (cons c acc))
                   (loop (+ i 1) 'string (cons c acc))))
              ;; ----- comentario de linha: pula ate a quebra -----
              ((eq? estado 'linha)
               (if (char=? c #\newline)
                   (loop (+ i 1) 'normal (cons c acc))
                   (loop (+ i 1) 'linha acc)))
              ;; ----- comentario de bloco: pula ate */ -----
              ((eq? estado 'bloco)
               (cond
                 ((and (char=? c #\*) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\/))
                  (loop (+ i 2) 'normal acc))
                 ((char=? c #\newline)
                  (loop (+ i 1) 'bloco (cons c acc)))   ; mantem a linha
                 (else
                  (loop (+ i 1) 'bloco acc))))
              (else
               (loop (+ i 1) 'normal acc))))))
    (loop 0 'normal '())))

### 1.3 Leitura de números, identificadores e strings

Cada leitor recebe a string `s`, a posição inicial `i` e o tamanho `n`, e
devolve a **posição final** (onde o token termina). Assim o tokenizador sabe de
onde continuar.

In [ ]:
;; le um numero (inteiro ou decimal); devolve a posicao final
(define (le-numero s i n)
  (cond
    ((>= i n) i)
    ((eh-digito? (string-ref s i)) (le-numero s (+ i 1) n))
    ((and (char=? (string-ref s i) #\.) (< (+ i 1) n)
          (eh-digito? (string-ref s (+ i 1))))
     (le-numero s (+ i 2) n))
    (else i)))

;; le um identificador (letras, digitos, underline); devolve a posicao final
(define (le-ident s i n)
  (if (and (< i n)
           (or (eh-letra? (string-ref s i))
               (eh-digito? (string-ref s i))))
      (le-ident s (+ i 1) n)
      i))

;; le uma string entre aspas; i aponta DEPOIS da aspa de abertura.
;; devolve a posicao logo apos a aspa de fechamento
(define (le-string s i n)
  (cond
    ((>= i n) n)
    ((char=? (string-ref s i) #\") (+ i 1))
    (else (le-string s (+ i 1) n))))

### 1.4 A função `tokeniza`

Junta tudo: primeiro remove os comentários, depois varre o texto montando a
lista de tokens. Quebras de linha apenas aumentam o contador de linha (os tokens
`NEWLINE`/`INDENT` da indentação ficam para o Dia 2). `mostra-tokens` é só uma
ajuda para visualizar o resultado.

In [ ]:
;; transforma o texto-fonte numa lista de tokens
(define (tokeniza fonte)
  (let* ((s (remove-comentarios fonte))
         (n (string-length s)))
    (define (scan i linha acc)
      (if (>= i n)
          (reverse acc)
          (let ((c (string-ref s i)))
            (cond
              ((char=? c #\newline)
               (scan (+ i 1) (+ linha 1) acc))
              ((eh-espaco? c)
               (scan (+ i 1) linha acc))
              ;; numero
              ((eh-digito? c)
               (let ((j (le-numero s i n)))
                 (scan j linha
                       (cons (faz-token 'NUMBER (substring s i j) linha) acc))))
              ;; string entre aspas
              ((char=? c #\")
               (let ((j (le-string s (+ i 1) n)))
                 (scan j linha
                       (cons (faz-token 'STRING (substring s (+ i 1) (- j 1)) linha) acc))))
              ;; palavra (identificador ou reservada)
              ((eh-letra? c)
               (let* ((j (le-ident s i n))
                      (palavra (substring s i j)))
                 (scan j linha
                       (cons (faz-token (classifica palavra) palavra linha) acc))))
              ;; pontuacao usada pela gramatica: , e :
              ((char=? c #\,)
               (scan (+ i 1) linha (cons (faz-token 'PONT "," linha) acc)))
              ((char=? c #\:)
               (scan (+ i 1) linha (cons (faz-token 'PONT ":" linha) acc)))
              ;; caractere desconhecido: por enquanto ignora (erros: Dia 3)
              (else
               (scan (+ i 1) linha acc))))))
    (scan 0 1 '())))

;; mostra os tokens, um por linha (so para visualizar)
(define (mostra-tokens tokens)
  (for-each
    (lambda (t)
      (display (tipo-tok t))   (display "  ")
      (display (lexema-tok t)) (display "   (linha ")
      (display (linha-tok t))  (display ")")
      (newline))
    tokens))

### 1.5 Testes manuais

Rodando célula a célula para conferir o lexer.

In [ ]:
;; exemplo simples: o comentario deve sumir
(mostra-tokens (tokeniza "10 g sal // isto e um comentario"))

In [ ]:
;; metadados com string: NOME -> KEYWORD, : -> PONT, "..." -> STRING
(mostra-tokens (tokeniza "NOME: \"OmeleteComQueijo\""))

In [ ]:
;; comentario de bloco no meio da instrucao (pode ate quebrar linha)
(mostra-tokens
  (tokeniza "ADICIONAR ovo /* passo
   critico */ EM tigela"))

## 2. Parser (Pessoa B — a fazer)

Recebe a lista de tokens de `tokeniza` e constrói a **AST** (árvore sintática),
seguindo a gramática `G = (V, Σ, P, S)` da especificação. Usa a interface
`tipo-tok` / `lexema-tok` / `linha-tok` definida na seção 0.

_Reservado para a Pessoa B._

## 3. Pretty-print + Validação (Pessoa C — a fazer)

Imprime a AST de forma legível e roda as verificações estáticas (declarar antes
de usar, soma de `USANDO` vs estoque, conflito de utensílio em `AO_MESMO_TEMPO`,
referência de sub-receita).

_Reservado para a Pessoa C._

## 4. Exemplos

Receitas de teste para o pipeline completo. Hoje (Dia 1) dá para ver os tokens.

In [ ]:
;; trecho de receita: bloco INGREDIENTES (sub-receita vem como STRING)
(define exemplo-ingredientes
  "INGREDIENTES
    2 unidade ovo
    200 ml leite
    1 unidade \"CaldaDeChocolate\"")

(mostra-tokens (tokeniza exemplo-ingredientes))